In [ ]:
import pandas as pd
from bs4 import BeautifulSoup
import requests
import re
import os
import time
from dotenv import load_dotenv
from google import genai
from urllib.parse import urljoin
import pypdf
import io
from pydantic import BaseModel
from typing import List
import json
from urllib.parse import urlparse, urljoin

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                         "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36"}

In [38]:
# point at the credential FILE (key stays in the file, never typed here)
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/Users/sihyunlee/Documents/GitHub/gatewayinitiative-nonprofit-dashboard/testing/lawrence-police-daily-logs-b5030731d4b1.json"

client = genai.Client(
    vertexai=True,
    project="lawrence-police-daily-logs",
    location="us-central1"
)

In [ ]:
BATCH_SIZE = 10       # orgs per Gemini request (fewer requests = safer on limits)
TASK = "Summarize the organization's Strategic Plan (what they plan to do in the upcoming year) and their Impact Report (key metrics, accomplishments, and everything that happened in the past year)."
KEYWORDS = ["impact", "report", "strategic", "plan", "annual", "about", "results"]

In [ ]:
def extract_pdf_text(pdf_url):
    try:
        r = requests.get(pdf_url, headers=HEADERS, timeout=15)
        if r.status_code != 200:
            return ""
        pdf_file = io.BytesIO(r.content)          # treat the downloaded bytes like a file
        reader = pypdf.PdfReader(pdf_file)
        text = ""
        for page in reader.pages[:6]:             # first 6 pages is plenty for a summary
            page_text = page.extract_text()
            if page_text:
                text += page_text + " "
        return text
    except Exception:
        return ""

In [ ]:
def crawl_site(url):
    if not url.startswith("http"):
        return ""
 
    try:
        page = requests.get(url, headers=HEADERS, timeout=15)
        soup = BeautifulSoup(page.text, "html.parser")
    except Exception:
        return ""
 
    base_domain = urlparse(url).netloc   # e.g. "actlawrence.org", used to stay on-site
 
    good_links = []   # normal web pages worth reading
    pdf_links = []     # PDF files worth reading
 
    for link in soup.find_all("a"):
        href = link.get("href")
        if not href:
            continue
 
        full_url = urljoin(url, href)              # turn "/impact" into a full address
        label = link.get_text(" ", strip=True).lower()
 
        looks_useful = any(kw in href.lower() or kw in label for kw in KEYWORDS)
        if not looks_useful:
            continue
 
        if full_url.lower().endswith(".pdf"):
            pdf_links.append(full_url)
        elif urlparse(full_url).netloc == base_domain:
            good_links.append(full_url)
 
    good_links = list(set(good_links))[:6]
    pdf_links = list(set(pdf_links))[:3]     # a few PDFs is enough; more = slower + more tokens
 
    all_text = soup.get_text(" ", strip=True)
 
    # read each normal linked page
    for link in good_links:
        try:
            p = requests.get(link, headers=HEADERS, timeout=15)
            s = BeautifulSoup(p.text, "html.parser")
            all_text += " " + s.get_text(" ", strip=True)
        except Exception:
            continue
 
    # read each PDF's actual text
    for pdf_url in pdf_links:
        pdf_text = extract_pdf_text(pdf_url)
        if pdf_text:
            all_text += f" [FROM PDF {pdf_url}]: " + pdf_text
 
    return all_text

In [ ]:
def ask_gemini_batch(names, texts, task):
    prompt = f"For each organization below, do the following: {task}\n"
    prompt += "Use this EXACT format for each one:\n\n"
    prompt += "ORG [number]:\n"
    prompt += "Strategic Plan: [summary, or \"not mentioned\"]\n"
    prompt += "Impact Report: [summary, or \"not mentioned\"]\n\n"
 
    for n in range(len(names)):
        prompt += f"--- ORGANIZATION {n+1}: {names[n]} ---\n"
        prompt += texts[n][:6000] + "\n\n"   # cap per-org text so the batch stays a reasonable size
 
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    return response.text

In [ ]:
def parse_batch_answer(answer, batch_names):
    lines = answer.split("\n")
    batch_results = []
 
    for n in range(len(batch_names)):
        strategic_plan = "not found"
        impact_report = "not found"
        capture = False
 
        for line in lines:
            text = line.strip()
            if text.startswith(f"ORG {n+1}:"):
                capture = True
                continue
            if text.startswith("ORG ") and not text.startswith(f"ORG {n+1}:"):
                capture = False
            if capture:
                if text.lower().startswith("strategic plan:"):
                    strategic_plan = text.split(":", 1)[1].strip()
                elif text.lower().startswith("impact report:"):
                    impact_report = text.split(":", 1)[1].strip()
 
        batch_results.append({
            "Name": batch_names[n],
            "Strategic_Plan": strategic_plan,
            "Impact_Report": impact_report,
        })
    return batch_results

In [ ]:
df = pd.read_csv("GWIorgs_v4.csv")
df = df.fillna("")
 
# 1) crawl every org first (no Gemini calls yet)
print("Crawling all sites...")
names = []
texts = []
for i in range(len(df)):
    name = df.loc[i, "Name"]
    url = df.loc[i, "URL"]
    print(f"  [{i+1}/{len(df)}] {name}")
    try:
        text = crawl_site(url)
    except Exception:
        text = ""
    names.append(name)
    texts.append(text)
 
# 2) send to Gemini in batches
print("\nAsking Gemini in batches...")
results = []
for start in range(0, len(names), BATCH_SIZE):
    batch_names = names[start:start + BATCH_SIZE]
    batch_texts = texts[start:start + BATCH_SIZE]
 
    print(f"  batch starting at org {start+1}")
    try:
        answer = ask_gemini_batch(batch_names, batch_texts, TASK)
        results.extend(parse_batch_answer(answer, batch_names))
    except Exception as e:
        print(f"    batch failed: {e}")
        for name in batch_names:
            results.append({"Name": name,
                            "Strategic_Plan": "(request failed)",
                            "Impact_Report": "(request failed)"})
 
    time.sleep(5)
 
# 3) save
report = pd.DataFrame(results)
report.to_csv("gemini_plan_impact.csv", index=False)
print("\nDone! Saved to gemini_plan_impact.csv")

Crawling all sites...
  [1/65] ACT Lawrence
  [2/65] Bellesini Academy
  [3/65] Beyond Soccer
  [4/65] Bread and Roses Housing
  [5/65] Bread and Roses Kitchen
  [6/65] Children's Friend and Family Services- a division of JRI
  [7/65] Community Giving Tree
  [8/65] Community InRoads
  [9/65] EforAll
  [10/65] Elevated Thought
  [11/65] Elliot Community Services
  [12/65] Esperanza Academy
  [13/65] Essex County Habitat For Humanity
  [14/65] Family Services of Merrimack Valley/LMCC
  [15/65] Greater Lawrence Community Action Council (GLCAC)
  [16/65] Greater Lawrence Community Boating
  [17/65] Greater Lawrence Family Health
  [18/65] Greater Lawrence Fellowship of the Arts
  [19/65] Greater Lawrence Technical School
  [20/65] Groundwork Lawrence
  [21/65] Hands to Help
  [22/65] International Institute of Greater Lawrence
  [23/65] Jeanne Geiger Crisis Center
  [24/65] Lawrence Boys and Girls Club
  [25/65] Lawrence Catholic Academy
  [26/65] Lawrence Community Works
  [27/65] Lawrenc